## Libraries

In [1]:
import os

In [2]:
run = 'local' #local or cloud
os.environ["APP_ENV"] = run

In [3]:
import sys
sys.path.append('..')

In [4]:
from src.core_grass import*
%matplotlib inline
#%load_ext autoreload
#%autoreload 2

In [5]:
import subprocess
result = subprocess.run(["grass", "--config", "python_path"],
                        check=True, text=True, capture_output=True)
sys.path.append(result.stdout.strip())

# import GRASS Python packages
from collections import defaultdict
import grass.script as gs
import grass.jupyter as gj

# Parameters

In [6]:
#set project variables
Project_Area = 'haw_river'
projection = '2264' #set to None if you want to use the DEM's original projection
res = '1m'

## Set variables analysis
source = 'wbd' #wbd or user-defined
data_scale = 'HUC12' #other options HUC12, HUC10, HUC8 OR the field name of the source data to be split <-- case sensitive
select_data = '030300020407' #     # Value within the filter_field used for data selection 
to_headwaters = False #False if only interested in local huc area
input_shp_name = None #None or must be populated populated with a shapefile polygons if data_scale != HUCXX
buffer = 2000
auto_delete = False

In [7]:
if data_scale.find('HUC') >= 0:
    huc_level = re.findall("[0-9]+",data_scale)[0]
    if huc_level == str(len(select_data)):
        huc_digys = huc_level
    else:
        huc_digys = huc_level
        select_data = select_data[:int(huc_digys)]

In [8]:
## Paths:
parent_dir = pl.Path(os.getcwd()).parent/'data'
grass_data_dir = pl.Path(os.getcwd()).parent/'grassdata'
current_dir = pl.Path(os.getcwd())
outputs_dir =parent_dir/'Rasters'/Project_Area/'Outputs'
vector_dir = parent_dir/'Vectors'/Project_Area
raster_dir = parent_dir/'Rasters'/Project_Area
huc_path = vector_dir/'{}_aoi.geojson'.format(Project_Area)
git_data_repo = current_dir/'data_repository'
if input_shp_name:
    input_shp = vector_dir/input_shp_name
directory = outputs_dir

### Get area of interest

In [9]:
#read from to vector path from notebook 1
huc_path = vector_dir/'{0}_{1}_aoi.geojson'.format(Project_Area,select_data)
aoi = gpd.read_file(str(huc_path),driver="GeoJSON")

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GeoJSON does not support open option DRIVER
  return ogr_read(


### Start new grass session

In [10]:
# define GRASS data settings (adapt to your needs)
location = Project_Area+'_'+projection

In [11]:
# Create a project if it doesn't exist
if os.path.exists(grass_data_dir/location):
    pass
else:
    if len(projection) > 4:
        wkt_text = pyproj.CRS.from_authority("esri", int(projection)).to_wkt()
        with open(vector_dir/'{}_prj.wkt'.format(Project_Area), 'w') as t:
            t.write(wkt_text)
        gs.create_project(path=grass_data_dir,
                      name=location,
                      wkt=vector_dir/'{}_prj.wkt'.format(Project_Area))
    else:
        gs.create_project(path=grass_data_dir,
                  name=location,
                  epsg=projection)

In [12]:
# Start GRASS in default project mapset
session = gj.init(grass_data_dir/location)

#### Review existing files

In [13]:
#list all grass files
grass_maps = list_existing_grass(print_it=True)

Available vector maps:
aoi@PERMANENT
aoi_buffer@PERMANENT
site@PERMANENT
stream_030300020407@PERMANENT
tmp_bsn@PERMANENT
tmp_bsn_buffer@PERMANENT
v_carve_030300020407@PERMANENT

Available raster maps:
Alamance_QL1_2024_03ft_CountywideRaster_2026_01_05@PERMANENT
ExistingTopo_2026_07_02@PERMANENT
MASK@PERMANENT
accum_030300020407@PERMANENT
dem_030300020407@PERMANENT
dem_030300020407_b@PERMANENT
drain_dir_030300020407@PERMANENT
r_streams_030300020407@PERMANENT
r_twi_030300020407@PERMANENT
subbasins_030300020407@PERMANENT


In [14]:
basin_list = aoi[data_scale.lower()].to_list()
basins = 'aoi'
gs.parse_command('v.import', input= huc_path,  output= basins)
print('added aoi to grass')

added aoi to grass


In [16]:
#set variables
carved = True
outlet_point = vector_dir/f'v_{select_data}_outlet.shp'

In [17]:
#standard variables
threshold = 8000 #
dem = 'dem_'+str(select_data)
c_lines = f'v_carve_{select_data}'
accum = f'accum_{select_data}'
drain_dir = f'drain_dir_{select_data}'
r_basins = f'r_basins_{select_data}'
v_basins = f'v_basins_{select_data}'
sub_basins = f'subbasins_{select_data}'
v_streams = f'stream_{select_data}'
r_streams = f'r_streams_{select_data}'
twi = f'r_twi_{select_data}'
v_outlet = f'outlet_{select_data}'

In [18]:
if carved:
    dem = dem+'_b'

In [19]:
basins+'_buffer'

'aoi_buffer'

In [20]:
dem

'dem_030300020407_b'

In [21]:
#add mask and reset region
gs.run_command('v.buffer',input=basins,output=basins+'_buffer',distance = buffer)
#mask outside cells
gs.run_command('g.region',vector = basins+'_buffer',align=dem)
gs.run_command('r.mask',vector = basins+'_buffer')

gs.run_command('g.region',raster = dem,align=dem,zoom=dem)
reg = gs.parse_command('g.region',raster = dem,align=dem,zoom=dem,flags='gp')

In [22]:
cell_size = (float(reg.nsres)+ float(reg.ewres)) /2

## Watershed Delineation

In [22]:
#convert lines to points
from shapely.geometry import Point

gdf = gpd.read_file(vector_dir/'pipes.shp')
# Assuming 'gdf' is your existing GeoDataFrame with LineString geometries
gdf['geometry'] = gdf['geometry'].apply(lambda line: Point(line.coords[0]))

gdf.to_file(outlet_point)

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field INSTALLYEA created as String field, though DateTime requested.
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field LASTINSPEC created as String field, though DateTime requested.
  ogr_write(


In [23]:
#import outlet
gs.run_command('v.import', input= outlet_point,extent='region', output= v_outlet)

In [24]:
gs.run_command('r.stream.snap',input = v_outlet, output=f'{v_outlet}_snap',stream_rast = r_streams,radius = 3)

In [25]:
gs.run_command('v.db.connect', map=f'{v_outlet}_snap',table=v_outlet)

In [27]:
# query raster cells (a new column will be added to existing table)
gs.run_command('v.what.rast', map=f'{v_outlet}_snap', raster=accum, column='daa')

In [28]:
gs.run_command('v.out.ogr',input=f'{v_outlet}_snap',output=vector_dir/f'{v_outlet}_snap.geojson',type= 'point',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_outlet}_snap.geojson').to_file(vector_dir/f'{v_outlet}_snap.shp')

/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field INSTALLYEA created as String field, though DateTime requested.
  ogr_write(
/opt/conda/envs/grassenv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Field LASTINSPEC created as String field, though DateTime requested.
  ogr_write(


In [29]:
gs.run_command('r.stream.basins',points = f'{v_outlet}_snap', direction=drain_dir,basins=r_basins)

KeyboardInterrupt: 

In [73]:
gs.run_command('r.to.vect',input = r_basins,output = v_basins, type='area',column='basin_col', flags='s')

In [74]:
gs.run_command('v.out.ogr',input=v_basins,output=vector_dir/f'{v_basins}.geojson',type= 'area',format='GeoJSON')
#convert to shp
gpd.read_file(vector_dir/f'{v_basins}.geojson').to_file(vector_dir/f'{v_basins}.shp')

In [76]:
m = gj.InteractiveMap(height=600,width=1200)
m.add_vector(v_outlet)
m.add_vector(f'{v_outlet}_snap')
m.add_vector(v_basins)
m.show()

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## END